In [2]:
!mkdir -p agents

In [3]:
!ls

agents	sample_data


In [4]:
import getpass

GEMINI_API_KEY = getpass.getpass(
    "Enter Gemini API Key:AQ.Ab8RN6L91bTtyNave5NqA6PtfhOaclLft2DBMxtxVrWo1qaoZA "
)

Enter Gemini API Key:AQ.Ab8RN6L91bTtyNave5NqA6PtfhOaclLft2DBMxtxVrWo1qaoZA ··········


In [5]:
import os

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

In [6]:
!pip install -q -U google-genai pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 28.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.0 which is incompatible.


In [7]:
%%writefile agents/vision_agent.py

import os
import json
import re
from pathlib import Path

from PIL import Image
from google import genai


class VisionAgent:
    """
    GuardianAI Vision Agent

    Responsibilities:
    - Analyze emergency images
    - Identify visually observable features
    - Detect visible hazards
    - Return structured JSON

    This agent does NOT diagnose medical conditions.
    """

    def __init__(
        self,
        api_key=None,
        model="gemini-3.6-flash"
    ):
        """
        Initialize Vision Agent.

        Parameters
        ----------
        api_key : str, optional
            Gemini API key. If not provided, reads
            GEMINI_API_KEY from environment variables.

        model : str
            Gemini model used for image analysis.
        """

        self.api_key = (
            api_key
            or os.getenv("GEMINI_API_KEY")
        )

        if not self.api_key:
            raise ValueError(
                "GEMINI_API_KEY was not found. "
                "Set it as an environment variable."
            )

        self.model = model

        self.client = genai.Client(
            api_key=self.api_key
        )

    # =========================================================
    # PROMPT
    # =========================================================

    def _build_prompt(self):
        """
        Creates the system instructions for the Vision Agent.
        """

        return """
You are GuardianAI's Vision Agent.

Your responsibility is to analyze an emergency image
and identify ONLY visually observable information.

IMPORTANT SAFETY RULES:

1. Do NOT diagnose medical conditions.
2. Do NOT claim internal injuries.
3. Do NOT claim that a person definitely has a disease
   or medical condition.
4. Do NOT invent information.
5. Only report information reasonably supported by
   the image.
6. If something is unclear, mark it as false or uncertain.
7. Your output will be passed to a Medical Triage Agent.
8. The Medical Triage Agent will make the emergency
   severity assessment.

Analyze the following visual categories:

SCENE:
- road accident
- vehicle accident
- fire
- indoor emergency
- outdoor emergency
- other

PERSON:
- person detected
- approximate visible position
- lying down
- standing
- sitting
- unclear

VISIBLE INJURY INDICATORS:
- visible bleeding
- visible wound
- possible burn appearance
- visible swelling
- visible deformity
- possible visible fracture appearance

ENVIRONMENTAL HAZARDS:
- fire
- smoke
- traffic
- damaged vehicle
- dangerous objects
- other hazards

Return ONLY valid JSON.

Use exactly this structure:

{
    "scene_type": "",
    "person_detected": false,
    "person_position": "",
    "visible_bleeding": false,
    "visible_wound": false,
    "possible_burn": false,
    "visible_swelling": false,
    "visible_deformity": false,
    "possible_visible_fracture": false,
    "fire_detected": false,
    "smoke_detected": false,
    "vehicle_damage": false,
    "hazards": [],
    "observations": []
}
"""

    # =========================================================
    # IMAGE LOADING
    # =========================================================

    def _load_image(self, image_input):
        """
        Load an image from:

        - file path
        - PIL Image object

        Returns
        -------
        PIL.Image.Image
        """

        if isinstance(image_input, Image.Image):
            return image_input

        if isinstance(image_input, str):
            image_path = Path(image_input)

            if not image_path.exists():
                raise FileNotFoundError(
                    f"Image not found: {image_input}"
                )

            return Image.open(image_path)

        raise TypeError(
            "image_input must be a file path "
            "or PIL Image object."
        )

    # =========================================================
    # JSON CLEANING
    # =========================================================

    def _parse_json(self, response_text):
        """
        Convert Gemini response into Python dictionary.
        """

        text = response_text.strip()

        # Remove markdown JSON fences
        text = re.sub(
            r"```json",
            "",
            text,
            flags=re.IGNORECASE
        )

        text = re.sub(
            r"```",
            "",
            text
        )

        text = text.strip()

        try:
            return json.loads(text)

        except json.JSONDecodeError as error:

            raise ValueError(
                "Gemini did not return valid JSON.\n\n"
                f"Raw response:\n{text}"
            ) from error

    # =========================================================
    # VALIDATE RESULT
    # =========================================================

    def _validate_result(self, result):
        """
        Makes sure the Vision Agent returns the expected fields.
        """

        required_fields = [
            "scene_type",
            "person_detected",
            "person_position",
            "visible_bleeding",
            "visible_wound",
            "possible_burn",
            "visible_swelling",
            "visible_deformity",
            "possible_visible_fracture",
            "fire_detected",
            "smoke_detected",
            "vehicle_damage",
            "hazards",
            "observations"
        ]

        for field in required_fields:

            if field not in result:
                result[field] = None

        return result

    # =========================================================
    # MAIN ANALYSIS
    # =========================================================

    def analyze_image(self, image_input):
        """
        Analyze an emergency image.

        Parameters
        ----------
        image_input :
            Image path or PIL Image.

        Returns
        -------
        dict
            Structured Vision Agent result.
        """

        image = self._load_image(image_input)

        prompt = self._build_prompt()

        response = self.client.models.generate_content(
            model=self.model,
            contents=[
                prompt,
                image
            ]
        )

        result = self._parse_json(
            response.text
        )

        result = self._validate_result(
            result
        )

        return result

    # =========================================================
    # PRETTY DISPLAY
    # =========================================================

    def display_result(self, result):
        """
        Display Vision Agent result in readable format.
        """

        print("\n")
        print("=" * 70)
        print("        GUARDIANAI — VISION AGENT")
        print("=" * 70)

        print(
            f"Scene Type: "
            f"{result.get('scene_type')}"
        )

        print(
            f"Person Detected: "
            f"{result.get('person_detected')}"
        )

        print(
            f"Person Position: "
            f"{result.get('person_position')}"
        )

        print(
            f"Visible Bleeding: "
            f"{result.get('visible_bleeding')}"
        )

        print(
            f"Visible Wound: "
            f"{result.get('visible_wound')}"
        )

        print(
            f"Possible Burn: "
            f"{result.get('possible_burn')}"
        )

        print(
            f"Visible Swelling: "
            f"{result.get('visible_swelling')}"
        )

        print(
            f"Visible Deformity: "
            f"{result.get('visible_deformity')}"
        )

        print(
            f"Possible Visible Fracture: "
            f"{result.get('possible_visible_fracture')}"
        )

        print(
            f"Fire Detected: "
            f"{result.get('fire_detected')}"
        )

        print(
            f"Smoke Detected: "
            f"{result.get('smoke_detected')}"
        )

        print(
            f"Vehicle Damage: "
            f"{result.get('vehicle_damage')}"
        )

        print("\nHazards:")

        for hazard in result.get(
            "hazards",
            []
        ):
            print(f"  • {hazard}")

        print("\nObservations:")

        for observation in result.get(
            "observations",
            []
        ):
            print(f"  • {observation}")

        print("=" * 70)

Writing agents/vision_agent.py


In [8]:
!cat agents/vision_agent.py


import os
import json
import re
from pathlib import Path

from PIL import Image
from google import genai


class VisionAgent:
    """
    GuardianAI Vision Agent

    Responsibilities:
    - Analyze emergency images
    - Identify visually observable features
    - Detect visible hazards
    - Return structured JSON

    This agent does NOT diagnose medical conditions.
    """

    def __init__(
        self,
        api_key=None,
        model="gemini-3.6-flash"
    ):
        """
        Initialize Vision Agent.

        Parameters
        ----------
        api_key : str, optional
            Gemini API key. If not provided, reads
            GEMINI_API_KEY from environment variables.

        model : str
            Gemini model used for image analysis.
        """

        self.api_key = (
            api_key
            or os.getenv("GEMINI_API_KEY")
        )

        if not self.api_key:
            raise ValueError(
                "GEMINI_API_KEY was not found. "
           

In [9]:
%%writefile test_vision_agent.py

import os
import json
from PIL import Image
from IPython.display import display

from agents.vision_agent import VisionAgent


# =========================================================
# 1. CHECK API KEY
# =========================================================

if not os.getenv("GEMINI_API_KEY"):

    raise ValueError(
        "GEMINI_API_KEY is not set."
    )


# =========================================================
# 2. CREATE VISION AGENT
# =========================================================

vision_agent = VisionAgent()

print("✅ Vision Agent initialized successfully!")


# =========================================================
# 3. UPLOAD IMAGE
# =========================================================

image_path = "/content/accident.jpg"
# =========================================================
# 4. DISPLAY IMAGE
# =========================================================

image = Image.open(image_path)

print(
    f"Image size: {image.size}"
)

display(image)


# =========================================================
# 5. ANALYZE IMAGE
# =========================================================

print("\nAnalyzing image...")

result = vision_agent.analyze_image(
    image
)


# =========================================================
# 6. DISPLAY RESULT
# =========================================================

vision_agent.display_result(
    result
)


# =========================================================
# 7. RAW JSON
# =========================================================

print("\nRAW JSON:")
print(
    json.dumps(
        result,
        indent=4
    )
)

Writing test_vision_agent.py


In [11]:
!python test_vision_agent.py

✅ Vision Agent initialized successfully!
Image size: (866, 1390)
<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=866x1390 at 0x7F7A4580F8C0>

Analyzing image...
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


        GUARDIANAI — VISION AGENT
Scene Type: other
Person Detected: True
Person Position: unclear
Visible Bleeding: True
Visible Wound: True
Possible Burn: False
Visible Swelling: False
Visible Deformity: False
Possible Visible Fracture: False
Fire Detected: False
Smoke Detected: False
Vehicle Damage: False

Hazards:

Observations:
  • Visible skin abrasion and scraping on the knee
  • Redness and visible blood/fluid on the surface of the scrape
  • A hand is visible holding the leg behind the knee

RAW JSON:
{
    "scene_type": 

In [12]:
!python test_vision_agent.py

✅ Vision Agent initialized successfully!
Image size: (866, 1390)
<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=866x1390 at 0x7C6CE3C0B8C0>

Analyzing image...
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


        GUARDIANAI — VISION AGENT
Scene Type: other
Person Detected: True
Person Position: unclear
Visible Bleeding: True
Visible Wound: True
Possible Burn: False
Visible Swelling: False
Visible Deformity: False
Possible Visible Fracture: False
Fire Detected: False
Smoke Detected: False
Vehicle Damage: False

Hazards:

Observations:
  • Close-up of a knee showing a red skin abrasion wound with visible blood.
  • A hand is visible holding the leg above the injured area.
  • Surrounding environment is not visible.

RAW JSON:
{
    "